```ipython
%cd models
from glob import glob
from itertools import groupby
from os import rename, remove
for i,j in groupby(glob('*_*.pth'), lambda x:x.split('_')[0]):
    rename(max(j,key=lambda x:int(x[:-4].split('_')[1])),i+'.pth')
```

In [ ]:
from fastai.vision.all import *
from torch.nn import *


class Dat:
    a = torch.from_numpy(np.memmap(f"../tmp/news2016zh.H", "h"))

    def __init__(s, w, b, n=1):
        s.a = s.a.as_strided([len(s.a) - w + 1, w], [1, 1])
        s.b = b
        s.n = n

    def __len__(s):
        return len(s.a) // (s.a.size(1) * s.b * s.n)
    
    def __iter__(s):
        for x in range(len(s)):
            x = s.a[torch.randint(len(s.a), [s.b])].to("cuda", torch.long)
            yield x, x


class View(Module):
    def __init__(s, k):
        super().__init__()
        s.k = k

    def forward(s, x):
        return x.view(-1, s.k)


class PyramidAutoEncoder(Sequential):
    def forward(s, x):
        n = int(np.log2(x.size(1)))
        t = []
        for _ in range(n):
            x = s[0](x)
            t.append(x)
        t = list(s[1](torch.cat(t)).split([len(t) for t in t]))
        x = None
        while t:
            x = s[2](t.pop() if x is None else torch.cat([x, t.pop()]))
        return x.view(n, -1, x.size(1))


class MultiCrossEntropyLoss(CrossEntropyLoss):
    def forward(s, x, t):
        return -x.log_softmax(2)[:, torch.arange(x.size(1), device="cuda"), t.flatten()].mean()


def Accuracy(i):
    def Accuracy(x, t):
        return accuracy(x[i], t)

    Accuracy.__name__ = str(int(i))
    return Accuracy

In [ ]:
m = Learner(
    DataLoaders(Dat(256, 8, 500), Dat(256, 8, 50000)),
    Sequential(
        Embedding(21482, 256),
        PyramidAutoEncoder(
            Sequential(View(512), Linear(512, 256, 0), BatchNorm1d(256, track_running_stats=0), Swish()),
            Sequential(Linear(256, 256, 0), BatchNorm1d(256, track_running_stats=0), Swish(), Linear(256, 256)),
            Sequential(BatchNorm1d(256, track_running_stats=0), Swish(), Linear(256, 512, 0), View(256)),
        ),
        Linear(256, 256, 0),
        BatchNorm1dFlat(256, track_running_stats=0),
        Swish(),
        Linear(256, 21482),
    ).cuda(),
    MultiCrossEntropyLoss(),
    metrics=map(Accuracy, range(8)),
    cbs=[ShowGraphCallback, SaveModelCallback(fname='modele', with_opt=1, reset_on_fit=0)],
)
m.load('modeld', 'cuda')
m.validate()
# m.lr_find()

In [ ]:
m.fine_tune(500)